# PySpark DataFrame Aggregations — Part 6
# `groupBy()`, `agg()`, `count()`, `sum()`, `avg()`, `min()`, `max()`

> **Learning style:** SQL aggregation is already familiar, so this chapter focuses on distributed DataFrame patterns, operational metrics, data-quality profiling, reconciliation, and large-data experiments.
>
> **Dataset theme:** E-commerce orders and event-processing metrics. Examples are written for PySpark. Synthetic data is for learning mechanics and performance—not for making real business conclusions.

# Part 1 — Understand aggregation before coding

## 1. What is aggregation?

Aggregation reduces many input rows into summary values. For example, a million order records can be summarized into revenue per country, average processing time per channel, or the earliest/latest event date per product category.

| order_id | country | category | amount |
|---:|---|---|---:|
| 1001 | India | Electronics | 250 |
| 1002 | India | Books | 100 |
| 1003 | UK | Electronics | 400 |
| 1004 | India | Electronics | 150 |

Grouping by country yields two summary rows: India has 3 records and amount total 500; UK has 1 record and amount total 400. The input grain and the meaning of the metric must be understood before interpreting the result.

## 2. Why aggregations matter in Data Engineering

- Daily volume monitoring: events/orders per date and source.
- Gold-layer rollups: country/day/category metrics from detailed Silver data.
- Data quality: null counts, minimum/maximum dates, unexpected ranges.
- Reconciliation: compare row counts and numeric totals between pipeline stages.
- Operational monitoring: average and maximum processing time by channel or job.
- Business reporting: order volume, units, discounts, average order amount.
- Anomaly detection: categories with unusually high/low totals.
- Capacity planning: data volume by source, partition, or processing window.

## 3. The six core methods

| Method | Purpose | Result |
|---|---|---|
| `groupBy()` | Define the grouping keys | `GroupedData` object |
| `agg()` | Specify one or more aggregate expressions | DataFrame of metrics |
| `count()` | Count rows or non-null column values, depending on usage | Count |
| `sum()` | Add non-null numeric values | Total |
| `avg()` | Average non-null numeric values | Mean |
| `min()` | Smallest non-null value | Minimum |
| `max()` | Largest non-null value | Maximum |

**Important difference:** `df.count()` is an action returning a Python integer for the total number of rows. `count("*")` and `count("column")` are aggregation expressions. `count("*")` counts rows; `count("column")` counts only rows where that column is non-null.

## 4. `groupBy()` versus `agg()`

```python
grouped = orders_df.groupBy("country")  # GroupedData, not final metrics yet

country_metrics = orders_df.groupBy("country").agg(
    count("*").alias("row_count"),
    sum("amount").alias("amount_total"),
    avg("amount").alias("average_amount"),
    min("amount").alias("minimum_amount"),
    max("amount").alias("maximum_amount")
)
```

- `groupBy("country")` chooses one group per country.
- `agg(...)` specifies metrics for each group.
- `count("*")` counts rows even when other fields are null.
- `sum` and `avg` ignore null numeric values; an all-null group may return null for sum.
- `alias(...)` assigns a clear output column name.

## 5. How distributed aggregation works

For a very large DataFrame, Spark can aggregate locally within input partitions, shuffle partial results when identical keys must meet, and merge those results into final groups. Exact physical execution depends on the optimizer and Spark version.

- `groupBy("country")` often has relatively few groups.
- `groupBy("customer_id")` may have hundreds of thousands or millions of groups.
- A heavily repeated key such as `"UNKNOWN"` can create skew.
- More grouping keys increase the number of possible combinations.
- Do not collect detailed records to the driver just to calculate totals; aggregate in Spark.

---

# Part 2 — Create the realistic learning dataset

## 1. Dataset fields

| Column | Meaning |
|---|---|
| `order_id` | Order identifier |
| `customer_id` | Customer identifier |
| `country`, `city` | Market dimensions |
| `category` | Product category |
| `channel` | MobileApp, Website, Partner |
| `status` | Delivered, Pending, Returned, Cancelled |
| `amount` | Order amount |
| `quantity` | Number of units in the order |
| `discount` | Recorded discount amount |
| `event_date` | Order/event date |
| `processing_seconds` | Processing duration in seconds |

## 2. Sample DataFrame setup

Run this once. All examples below reuse `orders_df`.

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, count, sum, avg, min, max, round,
    countDistinct, when, to_date, concat, lpad, date_add
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType
)

spark = SparkSession.builder.appName("PySparkAggregations").getOrCreate()

data = [
    (10001, "C001", "India", "Chennai", "Electronics", "MobileApp", "Delivered", 2499.0, 2, 150.0, "2026-10-01", 1.8),
    (10002, "C002", "India", "Bengaluru", "Books", "Website", "Delivered", 450.0, 3, 20.0, "2026-10-01", 2.1),
    (10003, "C001", "India", "Chennai", "Home", "MobileApp", "Returned", 1800.0, 1, 100.0, "2026-10-02", 3.4),
    (10004, "C003", "UK", "London", "Electronics", "Website", "Delivered", 899.0, 1, 0.0, "2026-10-02", 1.2),
    (10005, "C004", "India", "Chennai", "Fashion", "Partner", "Cancelled", 1200.0, 2, 100.0, "2026-10-03", 4.0),
    (10006, "C005", "Singapore", "Singapore", "Electronics", "MobileApp", "Delivered", 3200.0, 1, 200.0, "2026-10-03", 1.1),
    (10007, "C002", "India", "Bengaluru", "Books", "Website", "Pending", 300.0, 2, 0.0, "2026-10-04", 2.9),
    (10008, "C006", "UK", "Manchester", "Home", "MobileApp", "Delivered", 2200.0, 1, 120.0, "2026-10-04", 2.5),
    (10009, "C007", "India", "Chennai", "Electronics", "Partner", "Delivered", 5600.0, 2, 500.0, "2026-10-05", 1.5),
    (10010, "C008", "Singapore", "Singapore", "Fashion", "Website", "Returned", 700.0, 1, 50.0, "2026-10-05", 5.2),
    (10011, "C003", "UK", "London", "Books", "MobileApp", "Delivered", 650.0, 4, 25.0, "2026-10-06", 1.7),
    (10012, "C009", "India", "Hyderabad", "Home", "Website", "Delivered", 1500.0, 1, 75.0, "2026-10-06", 2.0),
    (10013, "C010", "India", "Chennai", "Fashion", "MobileApp", "Pending", 950.0, 2, 50.0, "2026-10-07", 3.1),
    (10014, "C005", "Singapore", "Singapore", "Electronics", "Partner", "Delivered", 4100.0, 2, 250.0, "2026-10-07", 1.4),
    (10015, "C011", "UK", "Manchester", "Home", "Website", "Cancelled", 800.0, 1, 0.0, "2026-10-08", 4.5),
    (10016, "C012", "India", "Bengaluru", "Books", "MobileApp", "Delivered", 550.0, 2, 30.0, "2026-10-08", 1.9),
    (10017, "C013", "India", "Hyderabad", "Electronics", "Website", "Delivered", 2700.0, 1, 100.0, "2026-10-09", 2.2),
    (10018, "C014", "Singapore", "Singapore", "Home", "MobileApp", "Pending", 1700.0, 1, 80.0, "2026-10-09", 3.8),
    (10019, "C015", "UK", "London", "Fashion", "Partner", "Delivered", 1300.0, 3, 150.0, "2026-10-10", 1.6),
    (10020, "C016", "India", "Chennai", "Electronics", "Website", "Delivered", 7200.0, 3, 600.0, "2026-10-10", 1.3)
]

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount", DoubleType(), True),
    StructField("event_date", StringType(), True),
    StructField("processing_seconds", DoubleType(), True)
])

orders_df = spark.createDataFrame(data, schema).withColumn(
    "event_date", to_date(col("event_date"), "yyyy-MM-dd")
)
orders_df.printSchema()
orders_df.show(5, truncate=False)
```

The explicit schema defines column types. `customer_id` is a string because it is an identifier, not a quantity. `to_date` converts date text into a date column. `DoubleType` keeps this teaching setup simple; financial systems often use `DecimalType` for fixed precision and scale.

## 3. Generate one million synthetic records

This creates values using Spark expressions instead of building a million Python tuples on the driver.

```python
N = 1_000_000

large_orders_df = spark.range(0, N).select(
    (col("id") + 1000000).cast("long").alias("order_id"),
    concat(lit("C"), lpad((col("id") % 100000).cast("string"), 6, "0")).alias("customer_id"),
    when((col("id") % 3) == 0, "India")
        .when((col("id") % 3) == 1, "UK")
        .otherwise("Singapore").alias("country"),
    when((col("id") % 4) == 0, "Electronics")
        .when((col("id") % 4) == 1, "Books")
        .when((col("id") % 4) == 2, "Home")
        .otherwise("Fashion").alias("category"),
    when((col("id") % 3) == 0, "MobileApp")
        .when((col("id") % 3) == 1, "Website")
        .otherwise("Partner").alias("channel"),
    when((col("id") % 5) == 0, "Returned")
        .when((col("id") % 5) == 1, "Pending")
        .otherwise("Delivered").alias("status"),
    ((col("id") % 10000) + 100).cast("double").alias("amount"),
    ((col("id") % 5) + 1).cast("int").alias("quantity"),
    ((col("id") % 500) / 10).cast("double").alias("discount"),
    date_add(to_date(lit("2026-01-01")), (col("id") % 365).cast("int")).alias("event_date"),
    ((col("id") % 100) / 10.0).cast("double").alias("processing_seconds")
)

print("Rows:", large_orders_df.count())
large_orders_df.groupBy("country").count().show()
```

This is a distributed synthetic dataset with patterned values, not real customer behaviour. Begin with one million rows. Increase to ten million only if the environment has sufficient resources. Use it to inspect execution plans, shuffle stages, and task balance.

---

# Part 3 — `groupBy()`: definition and 10 examples

## Definition and parameters

`groupBy(*cols)` groups rows by one or more column names or expressions and returns a `GroupedData` object. An aggregation such as `.count()` or `.agg(...)` is needed to produce final summary rows.

- `*cols`: grouping column names or column expressions.
- Multiple columns create groups for each observed combination.
- Null grouping keys can form a null group.
- High-cardinality keys can create many groups and increase shuffle/memory costs.

### Example 1 — Count rows by country

```python
orders_df.groupBy("country").count().show()
```

`groupBy("country")` creates one group per country; `count()` counts records in each group; `show()` triggers execution and displays results. **Use case:** market-level order-volume monitoring.

### Example 2 — Count by status

```python
orders_df.groupBy("status").count().show()
```

Each status becomes a group. The count shows how many records are delivered, pending, returned, or cancelled. **Use case:** operations dashboard. To calculate status percentages, compare these counts with the overall count.

### Example 3 — Group by country and category

```python
orders_df.groupBy("country", "category").count().show()
```

Two keys create groups such as India + Electronics and India + Books. Only combinations present in the input appear. **Use case:** product mix by market.

### Example 4 — Group by channel and status

```python
orders_df.groupBy("channel", "status").count().show()
```

This shows the status distribution for MobileApp, Website, and Partner channels. It can help reveal whether a channel has a high count of pending or cancelled records.

### Example 5 — Count by city and sort descending

```python
orders_df.groupBy("city").count().orderBy("count", ascending=False).show()
```

`orderBy` sorts the already aggregated result; `ascending=False` puts the largest count first. Sorting can require distributed work. **Use case:** identify the cities contributing most records.

### Example 6 — Count by event date

```python
orders_df.groupBy("event_date").count().orderBy("event_date").show()
```

The date is a `DateType`, so groups are formed by date values. Sorting by date makes the output chronological. **Use case:** daily source-to-target volume reconciliation.

### Example 7 — Group by a computed value band

```python
orders_df.groupBy(
    when(col("amount") >= 3000, "HighValue")
    .when(col("amount") >= 1000, "MidValue")
    .otherwise("LowValue").alias("value_band")
).count().show()
```

`when` classifies each row; `otherwise` assigns the fallback; `alias` labels the computed grouping key. This groups by an expression without adding a permanent column first. **Use case:** value-band monitoring.

### Example 8 — Group by date, country, and channel

```python
orders_df.groupBy("event_date", "country", "channel").count().show()
```

Each group is a unique date-country-channel combination. This creates a useful operational rollup, but more grouping keys mean more possible groups and potentially higher shuffle cost.

### Example 9 — Group the million-row dataset

```python
large_orders_df.groupBy("country", "category").count().show()
```

This runs the same logical grouping on distributed data. Inspect the Spark UI to see tasks and shuffle behaviour. Since there are few countries and categories, the final result remains small even though the input has a million rows.

### Example 10 — Filter before grouping

```python
orders_df.filter(col("status") == "Delivered") \
    .groupBy("country").count().show()
```

The filter restricts the input to delivered rows before grouping. The result counts delivered records per country. Filter early when it matches the metric definition; filtering cancelled/returned orders changes what the metric means.

**Best practices:** use only necessary keys; avoid grouping by unique IDs unless intended; inspect skew; filter when semantically correct; remember `groupBy()` alone is not the final metric DataFrame.

---

# Part 4 — `agg()`: definition and 10 examples

## Definition and parameters

`agg(*exprs)` calculates one or more aggregation expressions on a `GroupedData` object. It is useful when several metrics should be computed for each group.

- `*exprs`: expressions such as `count("*")`, `sum("amount")`, `avg("amount")`, `min("amount")`, `max("amount")`.
- `.alias("name")`: names the output column.
- `agg()` does not define groups by itself; use `df.groupBy(...).agg(...)`.

### Example 1 — Multiple metrics by country

```python
orders_df.groupBy("country").agg(
    count("*").alias("order_count"),
    sum("amount").alias("gross_order_value"),
    avg("amount").alias("average_order_value")
).show()
```

`count("*")` counts rows; `sum` totals non-null amounts; `avg` calculates the mean of non-null amounts. Aliases make the output dashboard-friendly. One grouped aggregation is easier to maintain than separate summaries joined back together.

### Example 2 — Min and max by category

```python
orders_df.groupBy("category").agg(
    min("amount").alias("smallest_order"),
    max("amount").alias("largest_order")
).show()
```

This returns the lowest and highest non-null order amounts in each category. It can surface extremes for investigation, but an extreme is not automatically invalid.

### Example 3 — Amount, units, and discount by channel

```python
orders_df.groupBy("channel").agg(
    sum("amount").alias("amount_sum"),
    sum("quantity").alias("units_ordered"),
    avg("discount").alias("average_discount")
).show()
```

These metrics have different units: money-like amount, units, and average discount. Do not label `sum(amount)` as net revenue unless status, discounts, refunds, and taxes are defined.

### Example 4 — Daily operations scorecard

```python
orders_df.groupBy("event_date").agg(
    count("*").alias("records"),
    sum("amount").alias("amount_total"),
    avg("processing_seconds").alias("avg_processing_seconds"),
    max("processing_seconds").alias("slowest_processing_seconds")
).orderBy("event_date").show()
```

This combines daily volume, amount, typical processing time, and slowest observed processing time. Average latency can hide long tails, so use percentiles too when tail latency matters.

### Example 5 — Country-status rollup

```python
orders_df.groupBy("country", "status").agg(
    count("*").alias("records"),
    sum("amount").alias("amount_sum"),
    avg("amount").alias("amount_avg"),
    min("event_date").alias("first_date"),
    max("event_date").alias("last_date")
).show()
```

Each country-status pair receives count, amount metrics, and date boundaries. `min`/`max` work on dates because dates have ordering. This helps monitor statuses and data coverage.

### Example 6 — Aggregate expressions, not only column names

```python
orders_df.groupBy("country").agg(
    sum(col("amount") - col("discount")).alias("amount_less_discount"),
    avg(col("amount") / col("quantity")).alias("avg_order_amount_per_unit")
).show()
```

The expressions calculate a row-level value before aggregation. Average of row-level amount/quantity is not necessarily equal to sum(amount)/sum(quantity); choose the formula that matches the business metric. Null operands can make a row expression null.

### Example 7 — Conditional status counts in one aggregation

```python
orders_df.groupBy("country").agg(
    count("*").alias("all_orders"),
    sum(when(col("status") == "Delivered", 1).otherwise(0)).alias("delivered"),
    sum(when(col("status") == "Cancelled", 1).otherwise(0)).alias("cancelled")
).show()
```

`when(..., 1).otherwise(0)` turns each row into an indicator; `sum` counts the indicators per country. This pattern is useful for success/failure metrics without separate groupBy jobs.

### Example 8 — Customer summary

```python
orders_df.groupBy("customer_id").agg(
    count("*").alias("orders_in_dataset"),
    sum("amount").alias("amount_in_dataset"),
    avg("amount").alias("avg_order_amount"),
    min("event_date").alias("first_seen_date"),
    max("event_date").alias("last_seen_date")
).orderBy("orders_in_dataset", ascending=False).show()
```

This summarizes records for each customer. The names say “in dataset” because the DataFrame may only contain a date window, not true lifetime history. Grouping by customer can produce many groups at scale.

### Example 9 — All core metrics on large data

```python
large_orders_df.groupBy("country").agg(
    count("*").alias("rows"),
    sum("amount").alias("amount_sum"),
    avg("amount").alias("amount_avg"),
    min("amount").alias("amount_min"),
    max("amount").alias("amount_max")
).show()
```

This calculates five metrics for each country across a million synthetic rows. Observe the Spark UI for aggregation and shuffle stages. Exact physical execution is optimizer-dependent.

### Example 10 — Create a Gold-layer daily rollup

```python
gold_daily_country_df = orders_df.groupBy("event_date", "country").agg(
    count("*").alias("order_records"),
    sum("amount").alias("amount_total"),
    sum("discount").alias("discount_total"),
    avg("processing_seconds").alias("avg_processing_seconds"),
    min("processing_seconds").alias("min_processing_seconds"),
    max("processing_seconds").alias("max_processing_seconds")
)
```

This creates a compact daily-country summary for a dashboard or Gold-layer table. A production definition must specify how late-arriving data, refunds, cancelled orders, and status changes affect historical totals.

**Best practices:** use clear aliases; group compatible metrics in one call; define the metric precisely; remember null semantics; inspect the physical plan for large jobs.

---

# Part 5 — `count()`: definition and 10 examples

## Definition

- `df.count()` returns the total row count as a Python integer and triggers execution.
- `count("*")` counts rows in an aggregation.
- `count("column")` counts non-null values in that column.
- `countDistinct("column")` counts distinct non-null values; it is different from ordinary `count`.

### Example 1 — Count every row

```python
total_rows = orders_df.count()
print(total_rows)
```

This returns one scalar integer, not a DataFrame. It triggers a Spark action. Avoid repeated full counts on a large uncached DataFrame if not necessary.

### Example 2 — Count rows by country

```python
orders_df.groupBy("country").count().show()
```

`GroupedData.count()` counts records in each country. The default output metric is named `count`. Use `agg(count("*").alias("row_count"))` for a custom name.

### Example 3 — Count by status with alias

```python
orders_df.groupBy("status").agg(count("*").alias("status_records")).show()
```

`count("*")` counts all rows in each status, even if another column is null. This is useful for operational state monitoring.

### Example 4 — Count non-null customer IDs

```python
orders_df.agg(count("customer_id").alias("non_null_customer_ids")).show()
```

This counts rows with a populated customer ID. It does not count unique customers: repeated IDs are counted repeatedly.

### Example 5 — Compare total rows and non-null amounts

```python
orders_df.agg(
    count("*").alias("all_rows"),
    count("amount").alias("rows_with_amount")
).show()
```

The first expression counts every row; the second excludes rows whose amount is null. The difference indicates missing amount values when the same DataFrame snapshot is used.

### Example 6 — Count by date and channel

```python
orders_df.groupBy("event_date", "channel").agg(
    count("*").alias("order_count")
).orderBy("event_date", "channel").show()
```

Each date-channel combination gets a record count. This is useful for daily ingestion or channel reconciliation.

### Example 7 — Count delivered rows only

```python
delivered_count = orders_df.filter(col("status") == "Delivered").count()
print(delivered_count)
```

The filter selects delivered rows and `count()` returns their total. The result is a scalar. This does not create one count per country; add `groupBy("country")` if that is required.

### Example 8 — Conditional counts in one row

```python
orders_df.agg(
    count("*").alias("all_orders"),
    sum(when(col("status") == "Delivered", 1).otherwise(0)).alias("delivered"),
    sum(when(col("status") == "Returned", 1).otherwise(0)).alias("returned"),
    sum(when(col("status") == "Cancelled", 1).otherwise(0)).alias("cancelled")
).show()
```

`count("*")` gives the total; indicator sums count selected statuses. This produces a one-row scorecard. It assumes each record has one status value.

### Example 9 — Count million-row data by country

```python
large_orders_df.groupBy("country").agg(count("*").alias("row_count")).show()
```

Spark counts distributed records and returns only a small summary. This is safer and more scalable than collecting all million rows and counting them in Python.

### Example 10 — Profile missing fields

```python
orders_df.agg(
    count("*").alias("total_rows"),
    count("customer_id").alias("rows_with_customer"),
    count("amount").alias("rows_with_amount"),
    count("event_date").alias("rows_with_date")
).show()
```

For each column, `total_rows - count(column)` is the null count if the same input snapshot is used. This pattern is a compact first-pass data-quality profile.

**Count pitfalls:** ordinary count is not distinct count; `count(column)` excludes nulls; `df.count()` triggers a job; avoid repeating full scans unnecessarily.

---

# Part 6 — `sum()`: definition and 10 examples

## Definition

`sum(column_or_expression)` adds non-null numeric values. It can be used for total amounts, units, discounts, or conditional indicator counts. A sum is not automatically revenue; define treatment of taxes, refunds, returns, discounts, and cancelled orders.

### Example 1 — Overall amount total

```python
orders_df.agg(sum("amount").alias("total_amount")).show()
```

This adds non-null amounts across all rows and returns a one-row DataFrame. An all-null input can yield null rather than zero; decide whether a zero default is appropriate.

### Example 2 — Total amount by country

```python
orders_df.groupBy("country").agg(sum("amount").alias("country_amount")).show()
```

Country is the grouping key; sum totals amounts in each country. Clarify whether all statuses should contribute to the amount metric.

### Example 3 — Amount by category and status

```python
orders_df.groupBy("category", "status").agg(sum("amount").alias("amount_total")).show()
```

Each category-status combination gets a separate total. This helps analyse amount associated with pending, returned, or cancelled records without assuming it is recognized revenue.

### Example 4 — Units by category

```python
orders_df.groupBy("category").agg(sum("quantity").alias("units_ordered")).show()
```

This totals item quantities, not order records. An order with three units contributes 3 to the sum but only 1 to a row count.

### Example 5 — Discount totals by country

```python
orders_df.groupBy("country").agg(sum("discount").alias("discount_total")).show()
```

This totals recorded discounts. Verify whether discount is stored per order, per item, or as a percentage before interpreting the total.

### Example 6 — Sum amount after discount per row

```python
orders_df.groupBy("country").agg(
    sum(col("amount") - col("discount")).alias("amount_less_discount")
).show()
```

The row-level expression subtracts discount from amount, then sum combines those differences by country. Null operands make the expression null for that row, which sum ignores. This is a simplified metric, not automatically accounting net revenue.

### Example 7 — Sum delivered amounts conditionally

```python
orders_df.groupBy("country").agg(
    sum(when(col("status") == "Delivered", col("amount")).otherwise(0.0))
        .alias("delivered_amount")
).show()
```

Delivered rows contribute their amount; other statuses contribute zero. Sum adds the result per country. If amount is null on a delivered row, that expression is null and is ignored by sum.

### Example 8 — Daily amount and discount totals

```python
orders_df.groupBy("event_date").agg(
    sum("amount").alias("daily_amount"),
    sum("discount").alias("daily_discount")
).orderBy("event_date").show()
```

The date defines daily groups. Both totals are calculated in one aggregation and can be used for daily monitoring or reconciliation.

### Example 9 — Sum on million-row data

```python
large_orders_df.groupBy("country").agg(
    sum("amount").alias("amount_sum"),
    sum("quantity").alias("units_sum")
).show()
```

Spark aggregates across partitions and combines partial sums; it does not need to transfer every input row to the driver. Inspect the Spark UI to understand stages and shuffle behaviour.

### Example 10 — Delivered-only daily market rollup

```python
delivered_rollup_df = orders_df.filter(
    col("status") == "Delivered"
).groupBy("event_date", "country").agg(
    sum("amount").alias("delivered_amount"),
    sum("quantity").alias("delivered_units"),
    sum("discount").alias("delivered_discount")
)
```

Filtering defines the population; grouping creates daily country-level groups; sums calculate amount, units, and discount only for delivered records. A production metric must also define treatment of later refunds and returns.

**Sum pitfalls:** nulls are ignored; all-null groups may produce null; validate numeric types; use decimal types when monetary precision requires it; define the business meaning of the total.

---

# Part 7 — `avg()`: definition and 10 examples

## Definition

`avg(column_or_expression)` calculates the arithmetic mean of non-null values. `mean()` is an alias.

Average = sum of non-null values / count of non-null values.

This differs from dividing by the total row count when some values are null.

### Example 1 — Average order amount overall

```python
orders_df.agg(avg("amount").alias("average_order_amount")).show()
```

This calculates the mean of non-null amounts across the DataFrame. Averages can hide extreme values, so consider min, max, and percentiles too.

### Example 2 — Average by country

```python
orders_df.groupBy("country").agg(avg("amount").alias("avg_amount")).show()
```

Each country gets its own average order amount. Differences may reflect product mix, order statuses, or customer behaviour, so avoid interpreting the average without context.

### Example 3 — Average by category

```python
orders_df.groupBy("category").agg(avg("amount").alias("avg_category_order_amount")).show()
```

This is average amount per input record in each category, assuming each row represents an order. It is not necessarily an average per customer or per item.

### Example 4 — Average processing time by channel

```python
orders_df.groupBy("channel").agg(
    avg("processing_seconds").alias("avg_processing_seconds")
).show()
```

This measures average recorded processing duration per channel. Averages can hide long-tail events; for production latency, consider percentiles and counts.

### Example 5 — Average units per order by category

```python
orders_df.groupBy("category").agg(avg("quantity").alias("avg_units_per_order")).show()
```

This averages quantity per input record. It is not the total units sold; use sum for that. Confirm that one row represents one order at the intended grain.

### Example 6 — Average discount by country and channel

```python
orders_df.groupBy("country", "channel").agg(avg("discount").alias("avg_discount")).show()
```

Each country-channel combination receives an average discount. Null discounts are excluded, so include a non-null count when the denominator matters.

### Example 7 — Average row-level amount per unit

```python
orders_df.groupBy("category").agg(
    avg(col("amount") / col("quantity")).alias("avg_row_amount_per_unit")
).show()
```

This computes amount/quantity for each row, then averages those ratios. It differs from `sum(amount) / sum(quantity)`, which weights by units. Make sure quantity cannot be zero and choose the formula matching the business question.

### Example 8 — Average delivered amount only

```python
orders_df.filter(col("status") == "Delivered").groupBy("country").agg(
    avg("amount").alias("avg_delivered_amount")
).show()
```

Only delivered rows participate. The filter explicitly defines the population and prevents other statuses from affecting the mean.

### Example 9 — Average on large data

```python
large_orders_df.groupBy("country", "category").agg(
    avg("amount").alias("avg_amount"),
    avg("processing_seconds").alias("avg_processing_seconds")
).show()
```

This calculates means per country-category over a million synthetic rows. Spark aggregates in distributed execution. Synthetic values are for performance experiments, not real business conclusions.

### Example 10 — Average with its denominator and total

```python
orders_df.groupBy("country").agg(
    count("amount").alias("non_null_amount_count"),
    sum("amount").alias("amount_sum"),
    avg("amount").alias("amount_avg")
).show()
```

The count shows how many non-null amounts contribute to the mean. For a group with at least one non-null value, average is approximately sum divided by that non-null count, subject to numeric precision.

**Average pitfalls:** nulls are excluded; averages can hide outliers; average of ratios differs from ratio of totals; include counts for context.

---

# Part 8 — `min()`: definition and 10 examples

## Definition

`min(column_or_expression)` returns the smallest non-null value for the DataFrame or each group. It works on ordered values such as numbers and dates. For strings, the result follows string ordering, not necessarily a business priority.

### Example 1 — Minimum amount overall

```python
orders_df.agg(min("amount").alias("minimum_amount")).show()
```

Returns the smallest non-null amount across the input. A low amount is not automatically invalid; check the business rules.

### Example 2 — Minimum amount by country

```python
orders_df.groupBy("country").agg(min("amount").alias("smallest_order_amount")).show()
```

Each country gets its lowest amount. This can support sanity checks or identify values worth investigating.

### Example 3 — Earliest date by country

```python
orders_df.groupBy("country").agg(min("event_date").alias("first_event_date")).show()
```

For a date column, min returns the earliest date in each country. Useful for checking data coverage.

### Example 4 — Fastest processing time by channel

```python
orders_df.groupBy("channel").agg(
    min("processing_seconds").alias("fastest_processing_seconds")
).show()
```

This is the best observed processing time, not typical performance. Pair it with average/max or percentiles.

### Example 5 — Minimum quantity by category

```python
orders_df.groupBy("category").agg(min("quantity").alias("minimum_units")).show()
```

Identifies the smallest quantity in each category. If quantity must be positive, a zero or negative minimum can trigger a quality check.

### Example 6 — Minimum delivered amount

```python
orders_df.filter(col("status") == "Delivered").groupBy("country").agg(
    min("amount").alias("min_delivered_amount")
).show()
```

Only delivered rows contribute. The filter defines the population, and min finds the smallest amount in that population per country.

### Example 7 — Minimum discount by category and channel

```python
orders_df.groupBy("category", "channel").agg(
    min("discount").alias("minimum_discount")
).show()
```

This finds the lowest discount for each category-channel combination. A zero may mean no discount; min alone does not tell how common it is.

### Example 8 — Overall date boundaries

```python
orders_df.agg(
    min("event_date").alias("earliest_date"),
    max("event_date").alias("latest_date")
).show()
```

This returns the overall earliest and latest dates. It helps validate date coverage, but does not prove every date inside the range is present.

### Example 9 — Minimum on million-row data

```python
large_orders_df.groupBy("country").agg(min("amount").alias("minimum_amount")).show()
```

Spark combines local minima across partitions to produce group minima. This avoids collecting every value to the driver.

### Example 10 — Flag groups below a lower bound

```python
orders_df.groupBy("category").agg(
    min("amount").alias("minimum_amount"),
    count("*").alias("row_count")
).filter(col("minimum_amount") < 0).show()
```

The grouped result includes a minimum and count, then filters categories with a negative minimum. The threshold is appropriate only if negative values are invalid in the domain; refunds or credit transactions may legitimately be negative.

**Minimum pitfalls:** ignores nulls; is not the same as the first record; string min follows ordering; a minimum alone does not reveal frequency.

---

# Part 9 — `max()`: definition and 10 examples

## Definition

`max(column_or_expression)` returns the largest non-null value for the DataFrame or each group. It can be applied to numeric values and dates, among other ordered types.

### Example 1 — Maximum amount overall

```python
orders_df.agg(max("amount").alias("maximum_amount")).show()
```

Finds the largest non-null amount. Investigate context before labelling it an anomaly.

### Example 2 — Maximum amount by country

```python
orders_df.groupBy("country").agg(max("amount").alias("largest_order_amount")).show()
```

Finds each country's largest order amount. Maximum is sensitive to outliers and is not a measure of typical order size.

### Example 3 — Latest date by country

```python
orders_df.groupBy("country").agg(max("event_date").alias("latest_event_date")).show()
```

Returns the latest observed date per country. This can help identify feeds that stopped arriving when compared with the expected processing date.

### Example 4 — Slowest processing time by channel

```python
orders_df.groupBy("channel").agg(
    max("processing_seconds").alias("slowest_processing_seconds")
).show()
```

The maximum identifies the worst observed duration. Use it to investigate outliers, but consider percentiles and record counts for a fuller picture.

### Example 5 — Maximum quantity by category

```python
orders_df.groupBy("category").agg(max("quantity").alias("max_units_per_order")).show()
```

Finds the largest quantity in any record for each category. It may reveal bulk orders or unexpected quantities.

### Example 6 — Maximum delivered amount

```python
orders_df.filter(col("status") == "Delivered").groupBy("country").agg(
    max("amount").alias("max_delivered_amount")
).show()
```

The filter restricts the population to delivered records. Max then finds the largest amount per country in that subset.

### Example 7 — Maximum discount by category

```python
orders_df.groupBy("category").agg(max("discount").alias("maximum_discount")).show()
```

Finds the largest recorded discount per category. Confirm whether the field represents currency, percentage, or another unit.

### Example 8 — Latest date and maximum amount together

```python
orders_df.groupBy("country").agg(
    max("event_date").alias("latest_date"),
    max("amount").alias("largest_amount")
).show()
```

These are independent aggregations. The largest amount may not belong to the record with the latest date. If you need the entire record associated with the maximum date, use a window or join-back pattern.

### Example 9 — Maximum on large data

```python
large_orders_df.groupBy("country", "category").agg(
    max("amount").alias("maximum_amount")
).show()
```

Spark computes maxima per group across distributed partitions. Use `explain("formatted")` and the Spark UI to inspect execution for large datasets.

### Example 10 — Flag groups over an upper bound

```python
orders_df.groupBy("category").agg(
    max("amount").alias("maximum_amount"),
    count("*").alias("row_count")
).filter(col("maximum_amount") > 5000).show()
```

The code finds the maximum and row count per category, then keeps groups whose maximum exceeds 5000. The threshold is illustrative; production thresholds should be grounded in documented rules or validated statistical monitoring.

**Maximum pitfalls:** ignores nulls; a maximum value does not identify the full row; high values may be valid; check units and business rules.

---

# Part 10 — Real Data Engineering patterns

## 1. Build a market scorecard

```python
market_scorecard_df = orders_df.groupBy("country").agg(
    count("*").alias("order_rows"),
    sum("amount").alias("amount_total"),
    avg("amount").alias("average_amount"),
    min("amount").alias("minimum_amount"),
    max("amount").alias("maximum_amount"),
    sum("quantity").alias("units_ordered"),
    avg("processing_seconds").alias("average_processing_seconds"),
    max("processing_seconds").alias("maximum_processing_seconds")
)
```

This combines volume, amount range, unit totals, and processing duration in one compact output. Confirm the input grain: if one row represents an order line, `count("*")` is a line count, not an order count.

## 2. Reconcile raw and curated stages

```python
raw_metrics = raw_df.agg(
    count("*").alias("raw_rows"),
    sum("amount").alias("raw_amount")
)

silver_metrics = silver_df.agg(
    count("*").alias("silver_rows"),
    sum("amount").alias("silver_amount")
)
```

These create one-row metric DataFrames. Compare them with a cross join or collect only the tiny aggregate results if appropriate. Matching counts/totals are useful checks but do not prove every record is correct.

## 3. Profile null values

```python
null_profile_df = orders_df.agg(
    count("*").alias("total_rows"),
    sum(when(col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
    sum(when(col("amount").isNull(), 1).otherwise(0)).alias("null_amounts"),
    sum(when(col("event_date").isNull(), 1).otherwise(0)).alias("null_event_dates")
)
```

Each `when` expression creates a 1 for a null and 0 otherwise; `sum` adds those indicators. This differs from `count(column)`, which counts non-null values. Expand this pattern for required columns in a quality pipeline.

## 4. Distinguish rows, customers, and units

```python
orders_df.groupBy("country").agg(
    count("*").alias("order_rows"),
    countDistinct("customer_id").alias("distinct_customers"),
    sum("quantity").alias("units_ordered")
).show()
```

- `count("*")` counts input records.
- `countDistinct("customer_id")` counts unique non-null customer IDs.
- `sum("quantity")` totals units.
These are different metrics. `countDistinct` is included as a supporting pattern because it prevents a common misunderstanding about `count()`.

## 5. Inspect the execution plan

```python
large_orders_df.groupBy("country").agg(
    sum("amount").alias("amount_sum")
).explain("formatted")
```

`explain("formatted")` prints the plan details supported by the Spark version. Look for aggregate operators and exchanges/shuffles. The exact physical plan may differ with Spark versions and optimizations.

---

# Part 11 — Performance experiments and cautions

## 1. Aggregate in Spark; do not collect detail rows

Preferred:
```python
large_orders_df.groupBy("country").agg(sum("amount")).show()
```

Avoid for large datasets:
```python
# rows = large_orders_df.collect()
```

`collect()` transfers all rows to the driver and can exhaust driver memory. A grouped summary usually returns far fewer rows.

## 2. Cardinality and skew

- Country: often low cardinality.
- Country + category: more groups, still often manageable.
- Customer ID: potentially hundreds of thousands or millions of groups.
- Unique order ID: can create nearly one group per row and may not summarize meaningfully.
- A highly frequent key such as `UNKNOWN` can cause skew.

Check distributions and Spark UI metrics before changing partitioning or applying salting. Salting is not a default fix; it must be justified and safely recombined.

## 3. Round at the correct point

```python
orders_df.groupBy("country").agg(
    round(avg("amount"), 2).alias("average_amount")
).show()
```

Rounding after aggregation is usually a presentation choice. Rounding every input before summing can change totals. Follow the metric specification.

## 4. Null behaviour

- `count("*")`: all rows.
- `count("amount")`: non-null amount rows.
- `sum("amount")`: ignores null amounts; an all-null group can yield null.
- `avg("amount")`: averages non-null amounts.
- `min` / `max`: ignore null values.
- `when(condition, 1).otherwise(0)` plus `sum`: conditional row count.

Decide whether missing values should be ignored, rejected, counted, or replaced in the production metric.

# Part 12 — Common mistakes

1. Calling `groupBy()` and expecting the final summary without an aggregation.
2. Confusing `df.count()` with `count("column")`.
3. Assuming `count("customer_id")` counts unique customers.
4. Calling order amount “revenue” without defining statuses, discounts, refunds, and tax.
5. Forgetting `avg` ignores null values.
6. Assuming max amount and max date belong to the same record.
7. Grouping by too many high-cardinality keys.
8. Collecting a large DataFrame to the driver.
9. Running multiple separate scans when one `agg()` can calculate related metrics.
10. Treating synthetic data as evidence about real business behaviour.
11. Forgetting input grain: an input row may be an event, order, or order line.
12. Rounding values too early.

# Part 13 — Quick reference

```python
from pyspark.sql.functions import count, sum, avg, min, max

# Group and count
df.groupBy("country").count()

# Multiple keys
df.groupBy("country", "category").count()

# Multiple metrics
df.groupBy("country").agg(
    count("*").alias("row_count"),
    sum("amount").alias("amount_total"),
    avg("amount").alias("amount_avg"),
    min("amount").alias("amount_min"),
    max("amount").alias("amount_max")
)

# Overall metrics
df.agg(
    count("*").alias("rows"),
    sum("amount").alias("amount_total"),
    avg("amount").alias("amount_avg"),
    min("amount").alias("amount_min"),
    max("amount").alias("amount_max")
)

# Conditional count
df.groupBy("country").agg(
    sum(when(col("status") == "Delivered", 1).otherwise(0))
        .alias("delivered_count")
)

# Large-data aggregation
large_df.groupBy("country").agg(
    sum("amount").alias("amount_total")
).show()
```

## Final mental model

- `groupBy()` defines groups.
- `agg()` defines the metrics per group.
- `count()` measures rows or non-null values depending on its form.
- `sum()` totals numeric values.
- `avg()` calculates the mean of non-null values.
- `min()` and `max()` find extremes, not the complete record containing an extreme.
- In Data Engineering, use aggregations for rollups, monitoring, quality profiling, reconciliation, and Gold-layer datasets.
- At scale, aggregate in Spark, keep grouping keys intentional, inspect skew and execution plans, and avoid collecting detailed records to the driver.
